# LeetCode 1372 — Longest ZigZag Path in a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/longest-zigzag-path-in-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Tree, Depth-First Search, Binary Tree, DP on Trees

Skeleton LeetCode:

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None): ...
class Solution:
    def longestZigZag(self, root: TreeNode | None) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa **đường zigzag**: bắt đầu ở một nút bất kỳ, chọn hướng, rồi **luân phiên** trái/phải; độ dài = số **cạnh** (số nút − 1).
2. Viết brute force "từ mỗi nút đi zigzag hai hướng" và hiểu vì sao nó `O(N·H)`.
3. Nắm hai kiểu DP trên cây: **top-down** (truyền xuống `(hướng vừa đi, độ dài)`) và **bottom-up / hậu thứ tự** (mỗi nút trả về `(left_len, right_len)`).
4. Biết xử lý cây sâu tới `5·10^4` tầng trong Python: tăng giới hạn đệ quy hoặc viết bản **lặp**.
5. Tự dựng `TreeNode` + hàm dựng cây level-order, và một mẹo **đánh số nút** để trace dễ đọc khi mọi giá trị đều bằng 1.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân. Một **đường ZigZag** được tạo như sau:
1. Chọn một nút bất kỳ và một hướng (trái hoặc phải).
2. Nếu hướng hiện tại là phải thì đi sang con phải, ngược lại đi sang con trái.
3. Đổi hướng (phải ↔ trái).
4. Lặp lại bước 2–3 cho tới khi không đi được nữa.

Độ dài zigzag = số nút đã thăm − 1 (một nút đơn có độ dài 0). Trả về độ dài zigzag **dài nhất** trong cây.

- **Ví dụ 1:** `root = [1,null,1,1,1,null,null,1,1,null,1,null,null,null,1]` → `3` (phải → trái → phải).
- **Ví dụ 2:** `root = [1,1,1,null,1,null,null,1,1,null,1]` → `4` (trái → phải → trái → phải).
- **Ví dụ 3:** `root = [1]` → `0`.

**Ràng buộc:** số nút trong `[1, 5·10^4]`; `1 <= Node.val <= 100`.

In [1]:
import random
import sys
import time
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = trống)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q, i = deque([root]), 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

EX1 = [1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1]
EX2 = [1, 1, 1, None, 1, None, None, 1, 1, None, 1]
EX3 = [1]
EXAMPLES = [(EX1, 3), (EX2, 4), (EX3, 0)]
sys.setrecursionlimit(200000)
for arr, want in EXAMPLES:
    print(arr, "-> mong đợi", want)

[1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1] -> mong đợi 3
[1, 1, 1, None, 1, None, None, 1, 1, None, 1] -> mong đợi 4
[1] -> mong đợi 0


## 3. Kiến thức nền

### 3.1 `TreeNode`, dựng cây level-order và đánh số nút

Vì đề cho mọi giá trị bằng 1, rất khó nhìn ra "nút nào". Mẹo: thay mỗi giá trị khác `None` bằng **số thứ tự** của nó trong mảng level-order — cấu trúc cây giữ nguyên, nhưng trace dễ đọc.

In [2]:
def label(arr):
    """Thay giá trị bằng số thứ tự 1, 2, 3, ... (giữ nguyên vị trí None)."""
    out, k = [], 0
    for v in arr:
        if v is None:
            out.append(None)
        else:
            k += 1
            out.append(k)
    return out

def show(root, prefix="", tag="root"):
    """In cây dạng thụt lề: L = con trái, R = con phải."""
    if root is None:
        return
    print(prefix + f"{tag}: {root.val}")
    show(root.left, prefix + "    ", "L")
    show(root.right, prefix + "    ", "R")

print(label(EX1))
show(build_tree(label(EX1)))

[1, None, 2, 3, 4, None, None, 5, 6, None, 7, None, None, None, 8]
root: 1
    R: 2
        L: 3
        R: 4
            L: 5
                R: 7
                    R: 8
            R: 6


### 3.2 Đệ quy trên cây: truyền xuống (top-down) và trả lên (bottom-up)

- **Top-down:** thông tin đi **từ cha xuống con** qua tham số (ví dụ độ sâu hiện tại).
- **Bottom-up:** con tính xong mới trả kết quả **lên cha** (ví dụ chiều cao cây con).

Bài này giải được bằng cả hai kiểu — một bài tập tốt để so sánh.

In [3]:
def depth_top_down(node, d=0):
    if node is None:
        return d - 1
    return max(depth_top_down(node.left, d + 1), depth_top_down(node.right, d + 1))

def height_bottom_up(node):
    if node is None:
        return -1
    return 1 + max(height_bottom_up(node.left), height_bottom_up(node.right))

r = build_tree(EX1)
print("độ sâu lớn nhất (top-down):", depth_top_down(r), "| chiều cao (bottom-up):", height_bottom_up(r))

độ sâu lớn nhất (top-down): 5 | chiều cao (bottom-up): 5


### 3.3 Trả về tuple nhiều giá trị và biến `nonlocal`

Hàm có thể trả `(a, b)` rồi gỡ ra `a, b = f(...)`. Để cập nhật đáp án toàn cục bên trong hàm lồng, dùng `nonlocal best` (hoặc một list `[0]`).

In [4]:
def outer():
    best = 0
    def visit(x):
        nonlocal best
        best = max(best, x)
        return x, x * 2
    a, b = visit(5)
    visit(3)
    return a, b, best

print(outer())

(5, 10, 5)


### 3.4 Giới hạn đệ quy và DFS bằng stack

Cây suy biến `5·10^4` tầng vượt giới hạn mặc định (~1000). Ta đã gọi `sys.setrecursionlimit(200000)` ở trên; nhưng cách an toàn nhất là **DFS bằng stack** — mỗi phần tử stack chứa đầy đủ "tham số" mà đệ quy lẽ ra truyền.

In [5]:
def count_nodes_iter(root):
    stack, cnt = [root], 0
    while stack:
        node = stack.pop()
        if node is None:
            continue
        cnt += 1
        stack.append(node.left)
        stack.append(node.right)
    return cnt

print("số nút EX1:", count_nodes_iter(build_tree(EX1)), "| EX2:", count_nodes_iter(build_tree(EX2)))

số nút EX1: 8 | EX2: 7


## 4. Ý tưởng nền tảng / trực giác

Cây Ví dụ 1 đã đánh số (từ mục 3.1):

```
  1
   \
    2
   / \
  3   4
     / \
    5   6
     \
      7
       \
        8
```

Thử bằng tay một số điểm bắt đầu:
- Từ `1` đi **phải**: `1 → 2` (phải), đổi hướng → trái: `2 → 3`, đổi → phải: `3` không có con phải ⇒ dừng. Độ dài 2.
- Từ `2` đi **phải**: `2 → 4` (phải) → trái `4 → 5` → phải `5 → 7` → trái: `7` không có con trái ⇒ dừng. Độ dài **3**.
- Từ `1` đi phải rồi muốn tiếp tục tới `4`? Không được: từ `2` phải đi **trái** (đổi hướng).
- Từ `4` đi phải: `4 → 6`, đổi trái: `6` không có con ⇒ 1.

Đáp án 3. Nhận xét then chốt: *độ dài zigzag kết thúc tại một nút chỉ phụ thuộc vào cha nó và hướng đi vào nút*:
- Nếu vào nút bằng bước **trái** và bước trước đó của cha là **phải** ⇒ nối dài thêm 1.
- Nếu hai bước cùng hướng ⇒ zigzag cũ bị gãy, bắt đầu zigzag mới từ cha: độ dài 1.

## 5. Cách 1 — Brute Force / baseline: từ mỗi nút, đi zigzag hai hướng

**Ý tưởng:** với mọi nút `u` và mọi hướng khởi đầu `d ∈ {trái, phải}`, mô phỏng đúng như đề: đi theo `d`, đổi hướng, lặp đến khi hết. Lấy max.

In [6]:
def walk(node, go_left):
    """Mô phỏng zigzag từ node, bước đầu tiên theo go_left. Trả về số cạnh."""
    length = 0
    while True:
        nxt = node.left if go_left else node.right
        if nxt is None:
            return length
        node, go_left, length = nxt, not go_left, length + 1

def zigzag_brute(root):
    best = 0
    stack = [root]
    while stack:
        u = stack.pop()
        if u is None:
            continue
        best = max(best, walk(u, True), walk(u, False))
        stack.append(u.left); stack.append(u.right)
    return best

for arr, want in EXAMPLES:
    print(arr, "->", zigzag_brute(build_tree(arr)), "(mong đợi", want, ")")

[1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1] -> 3 (mong đợi 3 )
[1, 1, 1, None, 1, None, None, 1, 1, None, 1] -> 4 (mong đợi 4 )
[1] -> 0 (mong đợi 0 )


**Độ phức tạp:** `N` điểm bắt đầu × mỗi lần đi tối đa `H` bước (chiều cao) ⇒ `O(N·H)`. Cây cân bằng `H ≈ log N` thì rất nhanh, nhưng cây "zigzag dài" (một đường zigzag `N` nút) cho `O(N²)` — với `N = 5·10^4` là `2,5·10^9` bước.

**Nút thắt:** zigzag bắt đầu từ `2` (đi phải) và zigzag bắt đầu từ `4` (đi trái) **trùng phần đuôi** `4 → 5 → 7`; ta đi lại phần chung nhiều lần. Cần **tái sử dụng** kết quả: DP.

In [7]:
# minh họa nút thắt: một đường zigzag dài N nút -> brute force đi ~N^2/2 bước
def zigzag_chain(n):
    root = TreeNode(1)
    cur, go_left = root, True
    for _ in range(n - 1):
        nxt = TreeNode(1)
        if go_left: cur.left = nxt
        else: cur.right = nxt
        cur, go_left = nxt, not go_left
    return root

for n in [500, 1000, 2000]:
    t0 = time.perf_counter()
    zigzag_brute(zigzag_chain(n))
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=500: 0.008s
n=1000: 0.023s
n=2000: 0.079s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (top-down):** truyền xuống mỗi nút cặp `(đi vào bằng hướng nào, độ dài zigzag kết thúc tại đây)`:
- Từ nút `u` (đã vào bằng hướng `dir`, độ dài `len`):
  - Đi sang **con trái**: nếu `dir == phải` thì độ dài mới `len + 1`, ngược lại (`dir == trái`, gãy) thì `1`.
  - Đi sang **con phải**: nếu `dir == trái` thì `len + 1`, ngược lại `1`.
- Đáp án = max mọi `len` gặp được. Mỗi nút thăm **một lần** ⇒ `O(N)`.

**Insight (bottom-up / hậu thứ tự):** mỗi nút `u` trả về
- `L(u)` = zigzag dài nhất **bắt đầu tại `u`** với bước đầu **sang trái** = `1 + R(u.left)` (nếu có con trái), ngược lại 0.
- `R(u)` = bắt đầu tại `u` sang phải = `1 + L(u.right)` (nếu có con phải), ngược lại 0.
- Đáp án = `max(L(u), R(u))` trên mọi `u`.

**Quy tắc "nếu… thì…":** *nếu bước tiếp theo đổi hướng thì cộng 1; nếu không đổi thì bắt đầu lại từ 1 (không phải 0 — vì cạnh vừa đi vẫn tính).*

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS top-down mang (hướng, độ dài)

**Từng đoạn:** `dfs(node, went_left, length)` — `went_left` cho biết ta vào `node` bằng bước trái hay phải. Gốc không có bước vào; ta khởi động bằng cách gọi trực tiếp hai con của gốc với độ dài 1.

In [8]:
def zigzag_topdown(root):
    best = 0
    def dfs(node, went_left, length):
        nonlocal best
        if node is None:
            return
        best = max(best, length)
        # sang trái: nối dài nếu vừa đi phải, ngược lại bắt đầu lại = 1
        dfs(node.left, True, length + 1 if not went_left else 1)
        # sang phải: nối dài nếu vừa đi trái
        dfs(node.right, False, length + 1 if went_left else 1)
    dfs(root.left, True, 1)
    dfs(root.right, False, 1)
    return best

for arr, want in EXAMPLES:
    print(arr, "->", zigzag_topdown(build_tree(arr)), "(mong đợi", want, ")")

[1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1] -> 3 (mong đợi 3 )
[1, 1, 1, None, 1, None, None, 1, 1, None, 1] -> 4 (mong đợi 4 )
[1] -> 0 (mong đợi 0 )


Code nộp LeetCode (Cách 2):

In [9]:
class Solution:
    def longestZigZag(self, root: "TreeNode | None") -> int:
        self.best = 0
        def dfs(node, went_left, length):
            if node is None:
                return
            self.best = max(self.best, length)
            dfs(node.left, True, 1 if went_left else length + 1)
            dfs(node.right, False, length + 1 if went_left else 1)
        dfs(root.left, True, 1)
        dfs(root.right, False, 1)
        return self.best

print([Solution().longestZigZag(build_tree(a)) for a, _ in EXAMPLES])

[3, 4, 0]


**Độ phức tạp Cách 2:** `O(N)` thời gian (mỗi nút một lần), `O(H)` bộ nhớ ngăn xếp.

### 7.2 Cách 3 — DP hậu thứ tự trả `(left_len, right_len)`

**Từng đoạn:**
1. `post(None)` trả `(-1, -1)` — mẹo: khi đó `1 + (-1) = 0`, tự động xử lý "không có con".
2. `post(u)`: lấy `(lL, lR) = post(u.left)`, `(rL, rR) = post(u.right)`.
3. `L = 1 + lR` (sang trái rồi tiếp tục sang phải từ con trái), `R = 1 + rL`.
4. Cập nhật `best = max(best, L, R)`.

In [10]:
def zigzag_postorder(root):
    best = 0
    def post(node):
        nonlocal best
        if node is None:
            return -1, -1
        _, l_right = post(node.left)
        r_left, _ = post(node.right)
        L = 1 + l_right          # bắt đầu tại node, đi trái trước
        R = 1 + r_left           # bắt đầu tại node, đi phải trước
        best = max(best, L, R)
        return L, R
    post(root)
    return best

for arr, want in EXAMPLES:
    print(arr, "->", zigzag_postorder(build_tree(arr)), "(mong đợi", want, ")")

[1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1] -> 3 (mong đợi 3 )
[1, 1, 1, None, 1, None, None, 1, 1, None, 1] -> 4 (mong đợi 4 )
[1] -> 0 (mong đợi 0 )


Code nộp LeetCode (Cách 3):

In [11]:
class Solution:
    def longestZigZag(self, root: "TreeNode | None") -> int:
        best = 0
        def post(node):
            nonlocal best
            if node is None:
                return -1, -1
            _, lr = post(node.left)
            rl, _ = post(node.right)
            L, R = lr + 1, rl + 1
            best = max(best, L, R)
            return L, R
        post(root)
        return best

print([Solution().longestZigZag(build_tree(a)) for a, _ in EXAMPLES])

[3, 4, 0]


**Độ phức tạp Cách 3:** `O(N)` thời gian, `O(H)` bộ nhớ.

### 7.3 Cách 4 — Bản lặp (stack) của top-down: an toàn với cây sâu 5·10^4

Mỗi phần tử stack là bộ ba `(nút, vào_bằng_trái, độ dài)`. Gốc được đẩy với độ dài 0 và một "hướng giả"; để không làm sai, ta lưu **hai** độ dài: `lft` = zigzag kết thúc tại nút với bước cuối là trái, `rgt` = bước cuối là phải.

In [12]:
def zigzag_iter(root):
    best = 0
    stack = [(root, 0, 0)]       # (node, lft, rgt)
    while stack:
        node, lft, rgt = stack.pop()
        best = max(best, lft, rgt)
        if node.left:
            # vào con trái: zigzag nối từ zigzag kết thúc bằng bước phải tại node
            stack.append((node.left, rgt + 1, 0))
        if node.right:
            stack.append((node.right, 0, lft + 1))
    return best

for arr, want in EXAMPLES:
    print(arr, "->", zigzag_iter(build_tree(arr)), "(mong đợi", want, ")")

[1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1] -> 3 (mong đợi 3 )
[1, 1, 1, None, 1, None, None, 1, 1, None, 1] -> 4 (mong đợi 4 )
[1] -> 0 (mong đợi 0 )


Giải thích: tại con trái, zigzag kết thúc bằng bước **trái** có độ dài `rgt(cha) + 1` (nối vào zigzag cha kết thúc bằng bước phải; nếu cha có `rgt = 0` thì là 1 — tức bắt đầu mới). Zigzag kết thúc bằng bước **phải** tại con trái là không thể (bước cuối là trái) ⇒ 0. Gốc có cả hai bằng 0.

Code nộp LeetCode (Cách 4):

In [13]:
class Solution:
    def longestZigZag(self, root: "TreeNode | None") -> int:
        best = 0
        stack = [(root, 0, 0)]
        while stack:
            node, lft, rgt = stack.pop()
            if lft > best: best = lft
            if rgt > best: best = rgt
            if node.left:
                stack.append((node.left, rgt + 1, 0))
            if node.right:
                stack.append((node.right, 0, lft + 1))
        return best

print([Solution().longestZigZag(build_tree(a)) for a, _ in EXAMPLES])

[3, 4, 0]


**Độ phức tạp Cách 4:** `O(N)` thời gian, `O(N)` bộ nhớ stack worst-case (thường `O(H)`), không phụ thuộc giới hạn đệ quy.

## 8. TRACE từng bước

Trace Cách 4 trên Ví dụ 1 **đã đánh số** (giá trị = số thứ tự nút). Bảng trạng thái mong đợi:

| Nút | đến từ | lft | rgt | best sau bước |
|---|---|---|---|---|
| 1 | gốc | 0 | 0 | 0 |
| 2 | 1 (phải) | 0 | 1 | 1 |
| 4 | 2 (phải) | 0 | 1 | 1 (gãy: phải-phải) |
| 6 | 4 (phải) | 0 | 1 | 1 |
| 5 | 4 (trái) | 2 | 0 | 2 |
| 7 | 5 (phải) | 0 | 3 | **3** |
| 8 | 7 (phải) | 0 | 1 | 3 |
| 3 | 2 (trái) | 2 | 0 | 3 |

In [14]:
def trace_zigzag(arr):
    root = build_tree(label(arr))
    best = 0
    stack = [(root, 0, 0, "gốc")]
    print("nút | đến từ      | lft | rgt | best")
    while stack:
        node, lft, rgt, src = stack.pop()
        best = max(best, lft, rgt)
        print(f"{node.val:3d} | {src:11s} | {lft:3d} | {rgt:3d} | {best}")
        if node.left:
            stack.append((node.left, rgt + 1, 0, f"{node.val} (trái)"))
        if node.right:
            stack.append((node.right, 0, lft + 1, f"{node.val} (phải)"))
    return best

print("kết quả:", trace_zigzag(EX1))

nút | đến từ      | lft | rgt | best
  1 | gốc         |   0 |   0 | 0
  2 | 1 (phải)    |   0 |   1 | 1
  4 | 2 (phải)    |   0 |   1 | 1
  6 | 4 (phải)    |   0 |   1 | 1
  5 | 4 (trái)    |   2 |   0 | 2
  7 | 5 (phải)    |   0 |   3 | 3
  8 | 7 (phải)    |   0 |   1 | 3
  3 | 2 (trái)    |   2 |   0 | 3
kết quả: 3


Trace Cách 3 (hậu thứ tự) trên Ví dụ 2 — in `(L, R)` của từng nút khi nó **trả về** (con trước, cha sau):

In [15]:
def trace_post(arr):
    root = build_tree(label(arr))
    best = 0
    def post(node, depth):
        nonlocal best
        if node is None:
            return -1, -1
        _, lr = post(node.left, depth + 1)
        rl, _ = post(node.right, depth + 1)
        L, R = lr + 1, rl + 1
        best = max(best, L, R)
        print("    " * depth + f"nút {node.val}: L = 1 + R(trái) = {L}, R = 1 + L(phải) = {R}, best = {best}")
        return L, R
    post(root, 0)
    return best

print(label(EX2))
print("kết quả:", trace_post(EX2))

[1, 2, 3, None, 4, None, None, 5, 6, None, 7]
                nút 7: L = 1 + R(trái) = 0, R = 1 + L(phải) = 0, best = 0
            nút 5: L = 1 + R(trái) = 0, R = 1 + L(phải) = 1, best = 1
            nút 6: L = 1 + R(trái) = 0, R = 1 + L(phải) = 0, best = 1
        nút 4: L = 1 + R(trái) = 2, R = 1 + L(phải) = 1, best = 2
    nút 2: L = 1 + R(trái) = 0, R = 1 + L(phải) = 3, best = 3
    nút 3: L = 1 + R(trái) = 0, R = 1 + L(phải) = 0, best = 3
nút 1: L = 1 + R(trái) = 4, R = 1 + L(phải) = 1, best = 4
kết quả: 4


Đọc kết quả: nút 1 (gốc) có `L = 4` — zigzag `1 → 2 (trái) → 4 (phải) → 5 (trái) → 7 (phải)`. Mỗi `L`/`R` dùng lại kết quả của con, không đi lại.

## 9. Code nộp LeetCode

Bản khuyên dùng: Cách 4 (lặp, `O(N)`, không lo giới hạn đệ quy).

In [16]:
class Solution:
    def longestZigZag(self, root: "TreeNode | None") -> int:
        best = 0
        stack = [(root, 0, 0)]          # (nút, zigzag kết thúc bằng bước trái, ... bằng bước phải)
        while stack:
            node, lft, rgt = stack.pop()
            best = max(best, lft, rgt)
            if node.left:
                stack.append((node.left, rgt + 1, 0))
            if node.right:
                stack.append((node.right, 0, lft + 1))
        return best

for arr, want in EXAMPLES:
    got = Solution().longestZigZag(build_tree(arr))
    print(got, "OK" if got == want else "SAI")

3 OK
4 OK
0 OK


## 10. Vì sao thuật toán đúng

Định nghĩa cho mỗi nút `v`: `lft(v)` = độ dài zigzag dài nhất **kết thúc tại `v`** với bước cuối là trái (0 nếu `v` là con phải hoặc gốc); `rgt(v)` tương tự với bước cuối là phải.

**Bổ đề:** nếu `v` là con trái của `u` thì `lft(v) = rgt(u) + 1`, `rgt(v) = 0`; nếu `v` là con phải của `u` thì `rgt(v) = lft(u) + 1`, `lft(v) = 0`.

*Chứng minh:* một zigzag kết thúc tại `v` bằng bước trái phải đi qua cạnh `u → v`; phần trước đó (nếu có) là zigzag kết thúc tại `u` với bước cuối **phải** (vì phải luân phiên). Dài nhất là `rgt(u)`; nếu `rgt(u) = 0` thì zigzag chỉ gồm cạnh `u → v`, độ dài 1 = `0 + 1`. Bước cuối là phải thì không thể vì `v` là con trái.

Mọi zigzag đều kết thúc tại **một** nút nào đó với bước cuối trái hoặc phải ⇒ đáp án = `max(lft(v), rgt(v))` trên mọi `v`. Cách 2 và 4 tính đúng công thức bổ đề theo thứ tự cha trước con; Cách 3 là dạng "đối xứng" (bắt đầu tại `v` thay vì kết thúc tại `v`) và chứng minh tương tự.

## 11. Phân tích Time / Space

Gọi `N` = số nút (≤ `5·10^4`), `H` = chiều cao (từ `log N` tới `N`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute force từ mỗi nút | `O(N·H)`; worst `O(N²)` | `O(N)` stack duyệt |
| 2. Top-down đệ quy | `O(N)` | `O(H)` ngăn xếp đệ quy |
| 3. Hậu thứ tự `(L, R)` | `O(N)` | `O(H)` |
| 4. Top-down lặp | `O(N)` | `O(N)` worst, thường `O(H)` |

- Cây cân bằng: brute force `O(N log N)` cũng chạy được; cây "dây zigzag" thì chỉ các cách `O(N)` sống sót.
- Đệ quy với `H = 5·10^4` cần `sys.setrecursionlimit` lớn và có thể chạm giới hạn stack C — bản lặp an toàn hơn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
METHODS = {
    "brute": zigzag_brute,
    "topdown": zigzag_topdown,
    "postorder": zigzag_postorder,
    "iter": zigzag_iter,
    "submit": lambda r: Solution().longestZigZag(r),
}
edge = [
    ([1, 2], 1),                                     # một cạnh trái
    ([1, None, 2], 1),                               # một cạnh phải
    ([1, 2, 3], 1),                                  # hai con, không zigzag dài hơn 1
    ([1, 2, None, 3], 1),                            # trái-trái: gãy
    ([1, 2, None, None, 3], 2),                      # trái-phải
    ([1, 2, None, None, 3, 4], 3),                   # trái-phải-trái
    ([1, 2, 3, None, 4, 5, None], 2),                # hai nhánh đối xứng
]
for name, fn in METHODS.items():
    for arr, want in EXAMPLES + edge:
        got = fn(build_tree(arr))
        assert got == want, (name, arr, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 5 cách đúng trên 10 test


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [18]:
def random_tree(rng, n):
    nodes = [TreeNode(rng.randint(1, 100)) for _ in range(n)]
    for i in range(1, n):
        while True:
            p = nodes[rng.randrange(i)]
            side = rng.choice("LR")
            if side == "L" and p.left is None:
                p.left = nodes[i]; break
            if side == "R" and p.right is None:
                p.right = nodes[i]; break
    return nodes[0]

rng = random.Random(1372)
for _ in range(500):
    root = random_tree(rng, rng.randint(1, 60))
    res = {name: fn(root) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, res
# cây "dây zigzag" có đáp án biết trước
for n in [1, 2, 5, 37]:
    assert zigzag_iter(zigzag_chain(n)) == n - 1
print("500 cây ngẫu nhiên + dây zigzag: mọi cách khớp nhau")

500 cây ngẫu nhiên + dây zigzag: mọi cách khớp nhau


### 12.3 Test lớn theo ràng buộc (`N = 5·10^4`)

In [19]:
big_chain = zigzag_chain(50_000)          # đáp án 49999
line = TreeNode(1); cur = line
for _ in range(49_999):
    cur.right = TreeNode(1); cur = cur.right   # đường thẳng: đáp án 1
for name in ["topdown", "postorder", "iter", "submit"]:
    t0 = time.perf_counter()
    a, b = METHODS[name](big_chain), METHODS[name](line)
    print(f"{name:9s} -> {a}, {b} | {time.perf_counter() - t0:.3f}s")

topdown   -> 49999, 1 | 0.026s
postorder -> 49999, 1 | 0.024s
iter      -> 49999, 1 | 0.010s


submit    -> 49999, 1 | 0.011s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số **nút** thay vì số **cạnh**

In [20]:
def wrong_count_nodes(root):
    best = 0
    def dfs(node, went_left, length):
        nonlocal best
        if node is None:
            return
        best = max(best, length)
        dfs(node.left, True, length + 1 if not went_left else 2)
        dfs(node.right, False, length + 1 if went_left else 2)
    dfs(root, True, 1)          # SAI: coi một nút có độ dài 1
    return best

for arr, want in EXAMPLES:
    print(arr, "-> sai:", wrong_count_nodes(build_tree(arr)), "| đúng:", want)

[1, None, 1, 1, 1, None, None, 1, 1, None, 1, None, None, None, 1] -> sai: 4 | đúng: 3
[1, 1, 1, None, 1, None, None, 1, 1, None, 1] -> sai: 5 | đúng: 4
[1] -> sai: 1 | đúng: 0


**Vì sao sai:** đề định nghĩa độ dài = số nút − 1. Đếm nút cho `[1]` ra 1 thay vì 0, và mọi đáp án bị **dư 1**.

**Code đúng:** khởi động độ dài 0 ở gốc (hoặc 1 ở hai con của gốc), "bắt đầu lại" bằng 1 cạnh.

### Lỗi 2 — "Gãy" zigzag thì reset về 0 thay vì 1

In [21]:
def wrong_reset_zero(root):
    best = 0
    def dfs(node, went_left, length):
        nonlocal best
        if node is None:
            return
        best = max(best, length)
        dfs(node.left, True, length + 1 if not went_left else 0)    # SAI: 0
        dfs(node.right, False, length + 1 if went_left else 0)      # SAI: 0
    dfs(root.left, True, 1)
    dfs(root.right, False, 1)
    return best

arr = [1, 2, None, 3, None, None, 4]        # trái, trái, rồi phải: zigzag dài nhất = 2 (2 -> 3 -> 4)
print("sai:", wrong_reset_zero(build_tree(arr)), "| đúng:", zigzag_brute(build_tree(arr)))

sai: 1 | đúng: 2


**Vì sao sai:** khi đi trái hai lần liên tiếp `1 → 2 → 3`, zigzag mới **bắt đầu tại 2** và đã gồm cạnh `2 → 3` ⇒ độ dài 1. Reset về 0 làm mất cạnh đó, nên `2 → 3 → 4` chỉ được tính là 1 thay vì 2.

**Code đúng:** `1 if went_left else length + 1` cho con trái (và đối xứng cho con phải).

### Lỗi 3 — Chỉ xét zigzag bắt đầu từ **gốc**

In [22]:
def wrong_only_root(root):
    return max(walk(root, True), walk(root, False))   # SAI: bỏ qua các điểm bắt đầu khác

arr = [1, 2, None, 3, None, None, 4, 5]     # zigzag dài nhất bắt đầu ở nút 2
print("sai:", wrong_only_root(build_tree(arr)), "| đúng:", zigzag_iter(build_tree(arr)))

sai: 1 | đúng: 3


**Vì sao sai:** đề cho phép "chọn **bất kỳ** nút". Ở cây trên, từ gốc đi trái được `1 → 2` rồi phải từ 2 không có (2 chỉ có con trái) ⇒ 1; nhưng từ `2` đi trái `2 → 3 → 4 → 5` dài 3.

**Code đúng:** duyệt mọi nút (brute force) hoặc dùng DP có "bắt đầu lại".

### Lỗi 4 — Đệ quy trên cây sâu 5·10^4 với giới hạn mặc định

Mô phỏng giới hạn mặc định rồi bắt lỗi để in (không để exception văng ra):

In [23]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    print(zigzag_topdown(zigzag_chain(5000)))
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")
finally:
    sys.setrecursionlimit(old)
print("bản lặp:", zigzag_iter(zigzag_chain(5000)))

RecursionError: maximum recursion depth exceeded ...
bản lặp: 4999


**Code đúng:** dùng Cách 4 (lặp) hoặc đặt `sys.setrecursionlimit(10**6)` trước khi đệ quy.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường đi xuống bắt đầu ở **bất kỳ** nút, có ràng buộc giữa các bước liên tiếp" ⇒ **DP trên cây với trạng thái nhỏ** (ở đây: hướng bước cuối).
- Câu hỏi then chốt: *"Muốn tính cho nút này, tôi cần biết gì về cha (top-down) hoặc về con (bottom-up)?"* Ở đây: chỉ cần **một số** cho mỗi hướng.
- Mẹo trả `-1` cho nút `None` để `1 + (-1) = 0` — bớt được câu `if`.
- Cây tới `10^4–10^5` nút trong Python ⇒ ưu tiên DFS **lặp**.
- Khi giá trị nút vô nghĩa (toàn 1), hãy **đánh số nút** để debug/trace.

## 15. Biến thể của bài

1. **Trả về chính đường zigzag** (danh sách nút): lưu thêm "nút bắt đầu" cùng độ dài trong stack.
2. **Đường thẳng dài nhất** (chỉ trái hoặc chỉ phải liên tục): đổi quy tắc — cùng hướng thì `+1`, đổi hướng thì reset 1.
3. **Zigzag có trọng số**: tối đa tổng giá trị nút thay vì số cạnh — cùng khung, cộng `val` thay cho `1`.
4. **LeetCode 543 Diameter / 124 Max Path Sum**: đường có thể "gập" qua một nút — DP hậu thứ tự trả về nhánh tốt nhất.
5. **LeetCode 1026 / 2246**: DP trên cây mang theo min/max hoặc ký tự cha — cùng tư duy "trạng thái nhỏ truyền theo cạnh".

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | từ mỗi nút đi zigzag 2 hướng | `O(N·H)` (worst `O(N²)`) | `O(N)` | kiểm chứng, cây nhỏ |
| 2. Top-down đệ quy | truyền (hướng vào, độ dài) | `O(N)` | `O(H)` | dễ hiểu, phỏng vấn |
| 3. Hậu thứ tự | trả `(L, R)` bắt đầu tại nút | `O(N)` | `O(H)` | khi quen DP bottom-up |
| 4. Top-down lặp | stack `(nút, lft, rgt)` | `O(N)` | `O(N)` | **nộp bằng Python**, cây sâu |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `longestZigZag` của `[1,2,3,4,5,6,7]` (cây đầy đủ 3 tầng) và của `[1,None,2,3]` là bao nhiêu?

**Bài 2 (tính tay):** với cây đã đánh số `[1,2,3,None,4,5]`, tính `(L, R)` theo Cách 3 cho mọi nút (`None` trả `(-1,-1)`).

**Bài 3 (điền TODO):** hoàn thiện Cách 3 dạng một hàm trả về `(L, R, best_trong_cây_con)` — không dùng biến ngoài.

In [24]:
def zigzag_pure(node):
    """Bài 3 — trả về (L, R, best) của cây con gốc node. (Bài tập — chưa gọi)"""
    if node is None:
        return -1, -1, 0
    # TODO: gọi đệ quy hai con, tính L, R, best = max(best_trái, best_phải, L, R)
    pass

**Bài 4 (tự viết lại):** viết lại Cách 2 nhưng dùng **BFS** (`deque`) thay cho đệ quy; mỗi phần tử hàng đợi là `(nút, went_left, length)`.

**Bài 5 (mini challenge):** viết `longest_straight(root)` — độ dài (số cạnh) đường đi xuống dài nhất mà **mọi bước cùng hướng** (toàn trái hoặc toàn phải).

In [25]:
def longest_straight(root):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Cây đầy đủ 3 tầng: đi tối đa 2 cạnh | `[1,None,2,3]`: phải rồi trái | 2 và 2 |
| 2 | Tính lá trước | lá: `(0, 0)` | nút 2: `(0, 1+L(4)=1)`; nút 3: `(1+R(5)=1, 0)`; nút 1: `(1+R(2)=2, 1+L(3)=2)` |
| 3 | Gỡ `(lL, lR, lb)` và `(rL, rR, rb)` | `L = lR + 1`, `R = rL + 1` | `return L, R, max(lb, rb, L, R)` |
| 4 | Khởi động hàng đợi với hai con của gốc | `popleft()` thay vì đệ quy | giống hệt công thức Cách 2 |
| 5 | Cùng hướng thì `+1` | đổi hướng thì reset 1 | dùng `(nút, lft, rgt)`: con trái nhận `(lft+1, 0)` |

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ans1 = [zigzag_iter(build_tree(a)) for a in ([1, 2, 3, 4, 5, 6, 7], [1, None, 2, 3])]
print("Bài 1:", ans1)
assert ans1 == [2, 2]

# Bài 2: in (L, R) từng nút theo Cách 3 (giá trị = số thứ tự)
def all_LR(root):
    out = {}
    def post(node):
        if node is None:
            return -1, -1
        _, lr = post(node.left)
        rl, _ = post(node.right)
        out[node.val] = (lr + 1, rl + 1)
        return out[node.val]
    post(root)
    return out

ans2 = all_LR(build_tree([1, 2, 3, None, 4, 5]))
print("Bài 2:", ans2)
assert ans2 == {4: (0, 0), 2: (0, 1), 5: (0, 0), 3: (1, 0), 1: (2, 2)}

Bài 1: [2, 2]
Bài 2: {4: (0, 0), 2: (0, 1), 5: (0, 0), 3: (1, 0), 1: (2, 2)}


In [27]:
# Bài 3
def zigzag_pure_sol(node):
    if node is None:
        return -1, -1, 0
    _, lR, lb = zigzag_pure_sol(node.left)
    rL, _, rb = zigzag_pure_sol(node.right)
    L, R = lR + 1, rL + 1
    return L, R, max(lb, rb, L, R)

rng = random.Random(3)
for _ in range(300):
    r = random_tree(rng, rng.randint(1, 40))
    assert zigzag_pure_sol(r)[2] == zigzag_brute(r)
print("Bài 3: OK")

Bài 3: OK


In [28]:
# Bài 4
def zigzag_bfs(root):
    best = 0
    q = deque()
    if root.left: q.append((root.left, True, 1))
    if root.right: q.append((root.right, False, 1))
    while q:
        node, went_left, length = q.popleft()
        best = max(best, length)
        if node.left:
            q.append((node.left, True, 1 if went_left else length + 1))
        if node.right:
            q.append((node.right, False, length + 1 if went_left else 1))
    return best

rng = random.Random(4)
for _ in range(300):
    r = random_tree(rng, rng.randint(1, 40))
    assert zigzag_bfs(r) == zigzag_brute(r)
print("Bài 4: OK")

Bài 4: OK


In [29]:
# Bài 5
def longest_straight_sol(root):
    best = 0
    stack = [(root, 0, 0)]      # (nút, đường thẳng trái kết thúc tại nút, đường thẳng phải)
    while stack:
        node, lft, rgt = stack.pop()
        best = max(best, lft, rgt)
        if node.left:
            stack.append((node.left, lft + 1, 0))
        if node.right:
            stack.append((node.right, 0, rgt + 1))
    return best

def straight_brute(root):
    best, stack = 0, [root]
    while stack:
        u = stack.pop()
        if u is None: continue
        for go_left in (True, False):
            k, cur = 0, u
            while (cur.left if go_left else cur.right):
                cur = cur.left if go_left else cur.right; k += 1
            best = max(best, k)
        stack += [u.left, u.right]
    return best

assert longest_straight_sol(build_tree([1, 2, None, 3, None, 4])) == 3
assert longest_straight_sol(zigzag_chain(10)) == 1
rng = random.Random(5)
for _ in range(300):
    r = random_tree(rng, rng.randint(1, 40))
    assert longest_straight_sol(r) == straight_brute(r)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP trên cây theo cạnh (lặp):**

```python
best = 0
stack = [(root, 0, 0)]                 # (nút, trạng thái A, trạng thái B)
while stack:
    node, lft, rgt = stack.pop()
    best = max(best, lft, rgt)
    if node.left:  stack.append((node.left,  rgt + 1, 0))   # đổi hướng: nối dài
    if node.right: stack.append((node.right, 0, lft + 1))
return best
```

**Template hậu thứ tự:** `None → (-1, -1)`; `L = 1 + R(trái)`, `R = 1 + L(phải)`; `best = max(best, L, R)`.

**Checklist trước khi submit:**
- [ ] Độ dài là số **cạnh** — `[1]` phải trả 0.
- [ ] Gãy zigzag (cùng hướng) ⇒ bắt đầu lại bằng **1**, không phải 0.
- [ ] Xét zigzag bắt đầu ở **mọi** nút (DP tự lo nếu có "bắt đầu lại").
- [ ] Cây sâu `5·10^4`: dùng bản lặp hoặc tăng giới hạn đệ quy.
- [ ] Đã thử: dây zigzag dài (đáp án `N−1`) và đường thẳng (đáp án 1).
- [ ] Đúng chữ ký: `longestZigZag(self, root) -> int`.